In [1]:
!pip install cohere hnswlib unstructured -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 17.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.7/370.7 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import cohere
import uuid
import hnswlib
from typing import List, Dict
from unstructured.partition.html import partition_html
from unstructured.chunking.title import chunk_by_title

co = cohere.Client("API_KEY")

In [7]:
raw_documents = [
  {
  "title": "Text Embeddings",
  "url": "https://docs.cohere.com/docs/text-embeddings"},
  {
  "title": "Using the Cohere Chat API for Text Generation",
  "url": "https://docs.cohere.com/docs/chat-api"},
  {
  "title": "Reasoning Capabilities",
  "url": "https://docs.cohere.com/docs/reasoning"},
  {
  "title": "Introduction to Embeddings at Cohere",
  "url": "https://docs.cohere.com/docs/embeddings"}
]

In [12]:
class Vectorstore:
  """
  A class representing a collection of documents indexed into a vectorstore.

  Parameters:
  raw_documents (list): A list of dictionaries representing the sources of the raw documents. Each dictionary should have 'title' and 'url' keys.

  Attributes:
  raw_documents (list): A list of dictionaries representing the raw documents.
  docs (list): A list of dictionaries representing the chunked documents, with 'title', 'text', and 'url' keys.
  docs_embs (list): A list of the associated embeddings for the document chunks.
  docs_len (int): The number of document chunks in the collection.
  idx (hnswlib.Index): The index used for document retrieval.

  Methods:
  load_and_chunk(): Loads the data from the sources and partitions the HTML content into chunks.
  embed(): Embeds the document chunks using the Cohere API.
  index(): Indexes the document chunks for efficient retrieval.
  retrieve(): Retrieves document chunks based on the given query.
  """
  def __init__(self, raw_documents: List[Dict[str, str]]):
    self.raw_documents = raw_documents
    self.docs = []
    self.docs_embs = []
    self.retrieve_top_k = 10
    self.rerank_top_k = 3
    self.load_and_chunk()
    self.embed()
    self.index()

  def load_and_chunk(self) -> None:
    """
    Loads the text from source and chunks the HTML content.
    """
    print("Loading Documents...")

    for raw_document in self.raw_documents:
      elements = partition_html(url=raw_document["url"])
      chunks = chunk_by_title(elements)
      for chunk in chunks:
        self.docs.append({
            "title": raw_document['title'],
            "text": str(chunk),
            "url": raw_document["url"]
        })

  def embed(self) -> None:
    """
    Embed the document chunks using Cohere API
    """
    print("Embedding document chunks...")

    batch_size=90
    self.docs_len = len(self.docs)
    for i in range(0, self.docs_len, batch_size):
      batch = self.docs[i: min(i + batch_size, self.docs_len)]
      texts = [item["text"] for item in batch]
      docs_embs_batch = co.embed(texts=texts, model="embed-english-v3.0", input_type="search_document").embeddings
      self.docs_embs.extend(docs_embs_batch)

  def index(self) -> None:
    """
    Indexs the Document chunks for efficient retrieval
    """
    print("Indexing document chunks...")

    self.idx = hnswlib.Index(space="ip", dim=1024)
    self.idx.init_index(max_elements=self.docs_len, ef_construction=512, M=64)
    self.idx.add_items(self.docs_embs, list(range(len(self.docs_embs))))

    print(f"Indexing complete with {self.idx.get_current_count()} document chunks.")

  def retrieve(self, query: str) -> List[Dict[str, str]]:
    """
    Retrieves document chunks based on the given query.

    Parameters:
    query (str): The query to retrieve document chunks for.

    Returns:
    List[Dict[str, str]]: A list of dictionaries representing the retrieved document chunks, with 'title', 'text', and 'url' keys.
    """

    #Dense retrieval
    query_emb = co.embed(
      texts=[query], model="embed-english-v3.0", input_type="search_query"
    ).embeddings

    doc_ids = self.idx.knn_query(query_emb, k=self.retrieve_top_k)[0][0]

    # Reranking
    rank_fields = ["title", "text"] # We'll use the title and text fields for reranking

    docs_to_rerank = [self.docs[doc_id] for doc_id in doc_ids]

    rerank_results = co.rerank(
      query=query,
      documents=docs_to_rerank,
      top_n=self.rerank_top_k,
      model="rerank-english-v3.0",
      rank_fields=rank_fields
    )

    doc_ids_reranked = [doc_ids[result.index] for result in rerank_results.results]

    docs_retrieved = []
    for doc_id in doc_ids_reranked:
      docs_retrieved.append({
          "title": self.docs[doc_id]['title'],
          "text": self.docs[doc_id]['text'],
          "url": self.docs[doc_id]["url"]
      })
    return docs_retrieved


In [13]:
vectorstore = Vectorstore(raw_documents)

Loading Documents...
Embedding document chunks...
Indexing document chunks...
Indexing complete with 57 document chunks.


In [14]:
vectorstore.retrieve("Embedding model")

[{'title': 'Introduction to Embeddings at Cohere',
  'text': 'In the example below we use the embed-v5.0-pro model to generate embeddings for 3 phrases and compare them using a similarity function. The two similar phrases have a high similarity score, and the embeddings for two unrelated phrases have a low similarity score:\n\nPYTHON\n\nimport cohere\n\nimport numpy as np\n\nco = cohere.ClientV2(api_key="YOUR_API_KEY")\n\n# get the embeddings\n\nphrases = ["i love soup", "soup is my favorite", "london is far away"]\n\nmodel = "embed-v5.0-pro"',
  'url': 'https://docs.cohere.com/docs/embeddings'},
 {'title': 'Introduction to Embeddings at Cohere',
  'text': 'model="embed-v5.0-pro",\n\ninput_type="search_document",\n\nembedding_types=["int8"],\n\nYou can specify multiple embedding types in a single call. For example, the following call will return both int8 and float embeddings:\n\nPYTHON\n\nres = co.embed(\n\ntexts=phrases,\n\nmodel="embed-v5.0-pro",\n\ninput_type=input_type,\n\nembeddi

In [21]:
class Chatbot:

  def __init__(self, vectorstore: Vectorstore):
    """
    Initializes an instance of the Chatbot class.

    Parameters:
    vectorstore (Vectorstore): An instance of the Vectorstore class.
    """
    self.vectorstore = vectorstore
    self.conversation_id = str(uuid.uuid4())

  def run(self):
    """
    Runs the chatbot application.
    """
    while True:
      print(f"\n{'-'*100}\n")

      #Get the user message
      message = input("User: ")

      # Typing "quit" ends the conversation
      if message.lower() == "quit":
        print("Ending chat.")
        break

      #else:                   # If using Google Colab, remove this line to avoid printing the same thing twice
      # print(f"User: (message}") # If using Google Colab, remove this line to avoid printing the same thing twice
      prompt = f"""
      Analyze the user query and classify it into one of two categories:
      1. 'RETRIEVE': The query requires specific, up-to-date, factual, or proprietary knowledge (e.g., stats, company policies, deep technical steps).
      2. 'DIRECT_ANSWER': The query is generic chit-chat, a greeting, basic math, common coding syntax, or a request to summarize previous chat history.

      Query: "{message}"
      Output exactly one word: RETRIEVE or DIRECT_ANSWER.
      """

      # Generate search queries (if any)
      response = co.chat(message=prompt,
                        model="command-a-03-2025",
                        )

      # If there are search queries, retrieve document chunks and respond
      if response.text == "RETRIEVE":
        print("Retrieving information...", end="")

        #Retrieve document chunks for each query
        documents = []
        # for query in response.search_queries:
        documents.extend(self.vectorstore.retrieve(message))

        #Use document chunks to respond
        response = co.chat_stream(
            message=message,
            model="command-a-03-2025",
            documents=documents,
            conversation_id=self.conversation_id,
        )

      # If there is no search query, directly respond
      else:
        response = co.chat_stream(
            message=message,
            model="command-a-03-2025",
            conversation_id=self.conversation_id
        )
      # Print the chatbot response, citations, and documents
      print("\nChatbot:")
      citations = []
      cited_documents = []

      # Display response
      for event in response:
        if event.event_type == "text-generation":
          print(event.text, end="")
        elif event.event_type == "citation-generation":
          citations.extend(event.citations)
        elif event.event_type == "stream-end":
          cited_documents = event.response.documents

      # Display citations and source documents
      if citations:
        print("\n\nCITATIONS:")
        for citation in citations:
          print(citation)

        print("\nDOCUMENTS:")
        for document in cited_documents:
          print(document)

In [22]:
chatbot = Chatbot(vectorstore)

chatbot.run()


----------------------------------------------------------------------------------------------------

User: I want to know about embeddings
Retrieving information...
Chatbot:
Embeddings are a way to represent the meaning of texts, images, or information as a list of numbers. Using a simple comparison function, you can then calculate a similarity score for two embeddings to figure out whether two pieces of information are about similar things. Common use-cases for embeddings include semantic search, clustering, and classification.

Matryoshka learning creates embeddings with coarse-to-fine representation within a single vector; embed-v5.0-pro supports multiple output dimensions in the following values: [256,512,768,1024,1536,2048]. To access this, you specify the parameter output_dimension when creating the embeddings.

The Cohere embeddings platform supports compression. The Embed API features an embeddings_types parameter which allows the user to specify various ways of compressing t